In [1]:
!pip install -qq faiss-cpu
!pip install -qq transformers
!pip install -qq pandas
!pip install -qq numpy
!pip install -qq scikit-learn
!pip install -qq tqdm
#Tai du lieu
!gdown --id 1N7rk-kfnDFIGMeX0ROVTjKh71gcgx-7R

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import faiss
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1N7rk-kfnDFIGMeX0ROVTjKh71gcgx-7R
To: /content/2cls_spam_text_cls.csv
100% 486k/486k [00:00<00:00, 7.80MB/s]


In [2]:
dataset_path = "/content/2cls_spam_text_cls.csv"
df = pd.read_csv(dataset_path)

messages = df['Message'].values.tolist()
labels = df['Category'].values.tolist()

In [3]:
MODEL_NAME = "intfloat/multilingual-e5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')

model = model.to(device)
model.eval()

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise

In [4]:
def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[...,None].bool(),0.0) #Kiểm tra xem đâu ko phỉa là từ thật thì điền 0
    return last_hidden.sum(dim =1)/attention_mask.sum(dim=1)[..., None]

In [18]:
def get_embedding(texts,model,tokenizer,device, batch_size = 32):
    embeddings = []
    for i in tqdm(range(0,len(texts),batch_size),desc = "Generating embedding"):
        batch_texts = texts[i:i+batch_size]
        batch_texts_with_prefix = [f"passage:{text}" for text in batch_texts]
        batch_dict = tokenizer(batch_texts_with_prefix, max_length= 512, padding = True, truncation = True, return_tensors = 'pt')
        batch_dict = {k: v.to(device) for k,v in batch_dict.items()}

        with torch.no_grad():
            outputs = model(**batch_dict)
            batch_embedding = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
            batch_embedding = F.normalize(batch_embedding,p=2,dim = 1)
            embeddings.append(batch_embedding.cpu().numpy())
    return np.vstack(embeddings)

In [19]:
le = LabelEncoder()
y = le.fit_transform(labels)

X_embeddings = get_embedding(messages, model, tokenizer, device)
metadata = [{"index":i,"message":message, "label":label,"label_encoded":y[i]}for i, (message,label) in enumerate(zip(messages,labels))]


Generating embedding: 100%|██████████| 175/175 [00:23<00:00,  7.39it/s]


In [28]:
TEST_SIZE = 0.1
SEED = 42
train_indices, test_indices = train_test_split(range(len(messages)),test_size = TEST_SIZE, stratify=y,random_state = SEED)

X_train_emb = X_embeddings[train_indices]
X_test_emb =X_embeddings[test_indices]
train_metadata = [metadata[i] for i in train_indices]
test_metadata = [metadata[i] for i in test_indices]
y_test = y[test_indices]
embedding_dim = X_train_emb.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(X_train_emb.astype("float32"))

In [48]:
def classify_with_knn(query_text, model, tokenizer, device, index, train_metadata, k=1):
    query_with_prefix = f"query: {query_text}"
    batch_dict = tokenizer([query_with_prefix], max_length=512, padding = True, trungcation = True, return_tensors= 'pt')
    batch_dict = {k:v.to(device) for k,v in batch_dict.items()}
    with torch.no_grad():
        outputs = model(**batch_dict)
        query_embedding = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
        query_embedding = F.normalize(query_embedding,p=2,dim=1)
        query_embedding = query_embedding.cpu().numpy().astype("float32")
    scores, indices = index.search(query_embedding,k)
    predictions=[]
    neighbor_info = []
    for i in range(k):
        neighbor_idx = indices[0][i]
        neighbor_score = scores[0][i]
        neighbor_label = train_metadata[neighbor_idx]["label"]
        neighbor_message = train_metadata[neighbor_idx]["message"]

        predictions.append(neighbor_label)
        neighbor_info.append({
            "score": float(neighbor_score),
            "label": neighbor_label,
            "message": neighbor_message[:100] + "..." if len(neighbor_message)>100 else neighbor_message
        })
    unique_labels,counts = np.unique(predictions,return_counts = True)
    final_prediction = unique_labels[np.argmax(counts)]
    return final_prediction,neighbor_info

In [34]:
def evaluate_knn_accuracy(test_embeddings,test_labels,test_metadata,index,train_metadata,k_values= [1,3,5]):
    results = {}
    all_errors= {}
    for k in k_values:
        correct = 0
        total = len(test_embeddings)
        errors=[]
        for i in tqdm(range(total),desc = f"Evaluating k={k}"):
            query_embedding = test_embeddings[i:i+1].astype("float32")
            true_label = test_metadata[i]["label"]
            true_message = test_metadata[i]["message"]

            scores,indices = index.search(query_embedding,k)
            predictions= []
            neighbor_details = []
            for j in range(k):
                neighbor_idx = indices[0][j]
                neighbor_label = train_metadata[neighbor_idx]["label"]
                neighbor_message = train_metadata[neighbor_idx]["message"]
                neighbor_score = float(scores[0][j])
                predictions.append(neighbor_label)
                neighbor_details.append({
                    "label":neighbor_label,
                    "message": neighbor_message,
                    "score": neighbor_score
                }
                )
            unique_labels,counts = np.unique(predictions,return_counts = True)
            predicted_label = unique_labels[np.argmax(counts)]

            if predicted_label == true_label:
                correct+=1
            else:
                error_info = {
                    "index": i,
                    "original_index": test_metadata[i]['index'],
                    'message': true_message,
                    'predicted_label':predicted_label,
                    "neighbors": neighbor_details,
                    "label_distribution": {label:int(count) for label,count in zip(unique_labels, counts)}
                }
                errors.append(error_info)
        accuracy = correct/total
        error_count = total - correct
        results[k] = accuracy
        all_errors[k] = errors
        print(f"Accuracy with k={k}: {accuracy:.4f}")
        print(f"Number of errors with k={k}: {error_count}/{total} ({(error_count/total)*100:.2f}%)")
    return results,all_errors

In [36]:
%%time
print("Evaluating accuracy on test set...")
accuracy_results,error_results = evaluate_knn_accuracy(
    X_test_emb,
    y_test,
    test_metadata,
    index,
    train_metadata,
    k_values = [1,3,5]
)
print("\n" + "="*50)
print("ACCURACY RESULTS")
print("="*50)
for k,accuracy in accuracy_results.items():
    print(f"Top-{k} accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("="*50)

import json
from datetime import datetime
error_analysis = {
    "timestamp" : datetime.now().isoformat(),
    "model": MODEL_NAME,
    "test_size": len(X_test_emb),
    "accuracy_results": accuracy_results,
    "errors_by_k":{}
}

for k,errors in error_results.items():
    error_analysis["errors_by_k"][f"k_{k}"] = {
        "total_errors": len(errors),
        "error_rate":len(errors)/len(X_test_emb),
        "erros": errors
    }

print()
print(f"***Summary:")
for k, errors in error_results.items():
    print(f" k={k}: {len(errors)} errors out of {len(X_test_emb)} samples")

Evaluating accuracy on test set...


Evaluating k=1: 100%|██████████| 558/558 [00:00<00:00, 1158.71it/s]


Accuracy with k=1: 0.9875
Number of errors with k=1: 7/558 (1.25%)


Evaluating k=3: 100%|██████████| 558/558 [00:00<00:00, 1196.82it/s]


Accuracy with k=3: 0.9928
Number of errors with k=3: 4/558 (0.72%)


Evaluating k=5: 100%|██████████| 558/558 [00:00<00:00, 1076.55it/s]

Accuracy with k=5: 0.9892
Number of errors with k=5: 6/558 (1.08%)

ACCURACY RESULTS
Top-1 accuracy: 0.9875 (98.75%)
Top-3 accuracy: 0.9928 (99.28%)
Top-5 accuracy: 0.9892 (98.92%)

***Summary:
 k=1: 7 errors out of 558 samples
 k=3: 4 errors out of 558 samples
 k=5: 6 errors out of 558 samples
CPU times: user 1.38 s, sys: 11.7 ms, total: 1.39 s
Wall time: 1.49 s


In [46]:
def spam_classifier_pipeline(user_input, k=3):

    prediction,neighbors = classify_with_knn(user_input,model, tokenizer, device, index, train_metadata,k=k)
    print(f"***Prediction: {prediction.upper()}")
    print()

    print("***Top neighbors:")
    for i, neighbor in enumerate(neighbors, 1):
        print(f"{i}. Label: {neighbor["label"]} | Score: {neighbor["score"]:.4f}")
        print(f" Message: {neighbor["message"]}")
        print()

     # Count label distribution
    labels = [n["label"] for n in neighbors]
    label_counts = {label: labels.count(label) for label in set(labels)}

    return {
     "prediction": prediction,
     "neighbors": neighbors,
     "label_distribution":label_counts }

In [49]:
test_exp = ["I am actually thinking a way of doing something useful","FREE!! Click here to win \$1000 NOW! Limited time offer!",]
for i, example in enumerate(test_exp, 1):
    print(f"\n--- Example {i}: '{example}' ---")
    result = spam_classifier_pipeline(example, k=3)
user_text = "Win a free iPhone! Click here now!"
k_value = 5
result = spam_classifier_pipeline(user_text, k=k_value)

<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_359/2598000889.py:1: SyntaxWarning: invalid escape sequence '\$'
  test_exp = ["I am actually thinking a way of doing something useful","FREE!! Click here to win \$1000 NOW! Limited time offer!",]



--- Example 1: 'I am actually thinking a way of doing something useful' ---
***Prediction: HAM

***Top neighbors:
1. Label: ham | Score: 0.8421
 Message: Same. Wana plan a trip sometme then

2. Label: ham | Score: 0.8402
 Message: that would be good … I'll phone you tomo lunchtime, shall I, to organise something?

3. Label: ham | Score: 0.8344
 Message: See? I thought it all through


--- Example 2: 'FREE!! Click here to win \$1000 NOW! Limited time offer!' ---
***Prediction: SPAM

***Top neighbors:
1. Label: spam | Score: 0.8579
 Message: Win a £1000 cash prize or a prize worth £5000

2. Label: spam | Score: 0.8508
 Message: FREE MESSAGE Activate your 500 FREE Text Messages by replying to this message with the word FREE For...

3. Label: spam | Score: 0.8508
 Message: FREE MESSAGE Activate your 500 FREE Text Messages by replying to this message with the word FREE For...

***Prediction: SPAM

***Top neighbors:
1. Label: spam | Score: 0.8614
 Message: FREE entry into our £250 weekly co